In [1]:
from dotenv import load_dotenv
load_dotenv("./../.env")

True

In [2]:
# configurations
DATA_DIR = "data"
CHROMA_DIR = "./chroma_financial_db"
COLLECTION_NAME = "financial_docs"
EMBEDDING_MODEL = 'nomic-embed-text'
BASE_URL = 'http://localhost:11434'
NUM_CTX = 8192 # 向量化维度

In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=BASE_URL, num_ctx=NUM_CTX)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

In [4]:
processed_hashes = set()

In [10]:
existing_docs = vector_store.get(where={"file_hash": {"$ne": ""}}, include=['metadatas'])
processed_hashes = [m.get('file_hash') for m in existing_docs['metadatas'] if m.get('file_hash')]
processed_hashes = set(processed_hashes)
processed_hashes

{'068176ab665682096a79859d73b805c7b2fda05331e503f26d893b582353464e',
 '1dceed8775b7b56432618aff917e08f9fe9573921f397a79b14bad5ab31e469a',
 '2c84425b68ab497014a25423984b2b8edc62c5de9a0b32abc605d92814408df7',
 '3f24e07aadeef2a3d0f40febf932f3d75d9e679b51b2bdfb47c66a606e589691',
 '46106605128d8b0dfc668e25b7cc70bdfc68e87fa8ba5d7aad8b0456665d3db9',
 '51ab83179bff647b1b2521836748d83939671d40a1d954fad9df93cd720e5784',
 '552615e47708aa125f69140f69d6bbd4c45f99981e4ed384f2d727d0665da8a8',
 '5b582c59f1d1fa81d5112c9e0b9aa87afdbc9f42418daace257bc360e0f61ef7',
 '60258a17806063663ac83b5fb5422d22bec255cfecbec842c5039e08992e6a46',
 '6e5549c7b20b0fbc5f482397070a1e85cbf8643c801ff570903f52366b11154f',
 'a06e0bf6d1ba3aac6b82958c381af08b4d476677484170264f6ae8d64fc4fe67',
 'b19b1f176c5ceaf9e6ea52907013480eb17716962351f8fdfe0ad4b13bb33ed0',
 'bfb57cd34d8c3d9f650b54f4914d712f7d88d57aa7b374886e23101da3228356',
 'c08079bc14250c896f3ca151f9a72ecc1ddcb9ca8e5b021539e91af10fae5c4b',
 'c2e9e49341c71edbe785ae4908a0b3ca

In [6]:
from utils import page_ingest
from pathlib import Path
data_path = Path(DATA_DIR)
pdf_files = list(data_path.rglob("*.pdf"))
pdf_files

[WindowsPath('data/amazon/amazon 10-k 2023.pdf'),
 WindowsPath('data/amazon/amazon 10-k 2024.pdf'),
 WindowsPath('data/amazon/amazon 10-q q1 2024.pdf'),
 WindowsPath('data/amazon/amazon 10-q q1 2025.pdf'),
 WindowsPath('data/amazon/amazon 10-q q2 2024.pdf'),
 WindowsPath('data/amazon/amazon 10-q q2 2025.pdf'),
 WindowsPath('data/amazon/amazon 10-q q3 2024.pdf'),
 WindowsPath('data/apple/apple 10-k 2023.pdf'),
 WindowsPath('data/apple/apple 10-k 2024.pdf'),
 WindowsPath('data/apple/apple 10-q q1 2024.pdf'),
 WindowsPath('data/apple/apple 10-q q2 2024.pdf'),
 WindowsPath('data/apple/apple 10-q q4 2023.pdf'),
 WindowsPath('data/apple/apple 8-k q4 2023.pdf'),
 WindowsPath('data/google/google 10-k 2023.pdf'),
 WindowsPath('data/google/google 10-k 2024.pdf'),
 WindowsPath('data/google/google 10-q q1 2025.pdf'),
 WindowsPath('data/google/google 10-q q2 2024.pdf'),
 WindowsPath('data/google/google 10-q q2 2025.pdf'),
 WindowsPath('data/google/google 10-q q3 2024.pdf')]

In [7]:
from pathlib import WindowsPath


for f in pdf_files:
    if f == WindowsPath("data/google/google 10-q q2 2025.pdf"):
        print("got it!")
        break

got it!


Ingest page data into database

In [8]:
for path in pdf_files:
    if path == WindowsPath("data/google/google 10-q q2 2025.pdf"):
        print("This file has a page that length exceeds the context length, ignore!")
        continue
    pdf_path = Path(path)
    page_ingest.ingest_docs_in_vectordb(pdf_path=pdf_path, vector_store=vector_store, processed_hashes=processed_hashes)
    print(f'finished ingest file {path} into database')

Processing: amazon 10-k 2023.pdf


[INFO] 2026-03-12 19:19:06,054 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:19:06,060 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 19:19:06,060 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 19:19:06,154 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:19:06,156 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 19:19:06,156 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 19:19:06,192 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:19:06,199 [RapidOCR] download_file.py:60

finished ingest file data\amazon\amazon 10-k 2023.pdf into database
Processing: amazon 10-k 2024.pdf


[INFO] 2026-03-12 19:20:39,414 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:20:39,418 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 19:20:39,418 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 19:20:39,498 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:20:39,500 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 19:20:39,500 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 19:20:39,538 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:20:39,546 [RapidOCR] download_file.py:60

finished ingest file data\amazon\amazon 10-k 2024.pdf into database
Processing: amazon 10-q q1 2024.pdf


...

finished ingest file data\google\google 10-k 2024.pdf into database
Processing: google 10-q q1 2025.pdf


[INFO] 2026-03-12 19:27:36,914 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:27:36,919 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 19:27:36,919 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 19:27:36,998 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:27:37,000 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 19:27:37,000 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 19:27:37,042 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:27:37,050 [RapidOCR] download_file.py:60

finished ingest file data\google\google 10-q q1 2025.pdf into database
Processing: google 10-q q2 2024.pdf


[INFO] 2026-03-12 19:28:15,182 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:28:15,186 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 19:28:15,187 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 19:28:15,270 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:28:15,271 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 19:28:15,272 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 19:28:15,312 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 19:28:15,321 [RapidOCR] download_file.py:60

finished ingest file data\google\google 10-q q2 2024.pdf into database
This file has a page that length exceeds the context length, ignore!
Processing: google 10-q q3 2024.pdf
finished ingest file data\google\google 10-q q3 2024.pdf into database


In [9]:
vector_store._collection.count()

1211

In [13]:
page_ingest.ingest_docs_in_vectordb(pdf_path=Path("data/apple/apple 8-k q4 2023.pdf"),vector_store=vector_store, processed_hashes=processed_hashes)

Processing: apple 8-k q4 2023.pdf
[SKIP] already processed: data\apple\apple 8-k q4 2023.pdf


In [14]:
# 关键词搜索
vector_store.get(where={"company_name": "amazon"}, limit=3)

{'ids': ['051f2c18-3bd4-444e-bb2f-e2b8b690a832',
  '58e91bd6-26ad-43ba-af6f-f44c8b204318',
  '681db586-a33f-4634-a1db-8b8e323c9977'],
 'embeddings': None,
 'documents': ["## UNITED STATES\n\n## SECURITIES AND EXCHANGE COMMISSION\n\nWashington, D.C. 20549\n\n\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\n\nFORM 10-K\n\n\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\n\n(Mark One)\n\n- [x] ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the fiscal year ended December 31, 2023\n\nor\n\n- [ ] ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the transition period from            to             .\n\nCommission File No. 000-22513\n\n\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\n\n## AMAZON.COM, INC.\n\n(Exact name 

In [15]:
# 相似度搜索
results = vector_store.search("What is Apple's revenue for Q1 2024", search_type="similarity")
results

[Document(id='8c5d39cf-f7ad-4e08-bf09-caa0ec8d4b46', metadata={'file_hash': 'bfb57cd34d8c3d9f650b54f4914d712f7d88d57aa7b374886e23101da3228356', 'company_name': 'apple', 'fiscal_quarter': 'q4', 'page': 5, 'source_file': 'apple 8-k q4 2023.pdf', 'doc_type': '8-k', 'fiscal_year': 2023}, page_content="\n\n## Apple reports first quarter results\n\n## Services revenue reaches new all-time record\n\n## EPS up 16 percent to new all-time high\n\nCUPERTINO, CALIFORNIA - Apple  today announced financial results for its fiscal 2024 first quarter ended December 30, 2023. The Company posted quarterly revenue of $119.6 billion, up 2 percent year over year, and quarterly earnings per diluted share of $2.18, up 16 percent year over year. ®\n\n'Today Apple is reporting revenue growth for the December quarter fueled by iPhone sales, and an all-time revenue record in Services,' said Tim Cook, Apple's CEO. 'We are pleased to announce that our installed base of active devices has now surpassed 2.2 billion, 